# Fase V — Predicción de toxicidad sobre plaguicidas panameños

Aplica el modelo **GNN-GIN** entrenado sobre Tox21 al corpus de plaguicidas registrados
en Panamá (ingredientes activos MIDA + familias PubChem HID 72).

**Requisitos previos:**
- `make train-gin` — checkpoint en `outputs/models/best_gin_model.pt`
- `make build-panama-corpus` — `data/processed/panama_corpus.pt`

**Salidas:**
- `outputs/results/panama_predictions.csv` — 12 probabilidades por compuesto
- `outputs/reports/panama_pesticides_profile.csv` — perfil resumido con nivel de riesgo
- Gráficos en `outputs/panama/`

**Niveles de riesgo:**
- **ALTO**: P > 0.7
- **MODERADO**: 0.4 < P ≤ 0.7
- **BAJO**: P ≤ 0.4

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import display

from src.data.dataset import TASK_NAMES
from src.data.pubchem_api import MIDA_ACTIVE_INGREDIENTS
from src.models.gin import GINToxicity

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 6)})

CONFIG_PATH = ROOT / "config" / "config.yaml"
MODEL_PATH = ROOT / "outputs" / "models" / "best_gin_model.pt"
CORPUS_PATH = ROOT / "data" / "processed" / "panama_corpus.pt"
CIDS_CSV = ROOT / "data" / "raw" / "pubchem_panama_cids.csv"
GHS_CSV = ROOT / "data" / "raw" / "pubchem_ghs_labels.csv"
OUT_DIR = ROOT / "outputs" / "results"
REPORT_DIR = ROOT / "outputs" / "reports"
FIG_DIR = ROOT / "outputs" / "panama"
for d in (OUT_DIR, REPORT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Raíz: {ROOT}")
print(f"Dispositivo: {device}")

## 1. Cargar corpus panameño

In [ ]:
if not CORPUS_PATH.is_file():
    raise FileNotFoundError(
        f"No existe {CORPUS_PATH}. Ejecuta: make build-panama-corpus"
    )

try:
    corpus = torch.load(CORPUS_PATH, map_location="cpu", weights_only=False)
except TypeError:
    corpus = torch.load(CORPUS_PATH, map_location="cpu")

entries = corpus["entries"]
meta = corpus.get("meta", {})
print(f"Grafos en corpus: {len(entries)}")
print(f"Meta: {meta}")

if CIDS_CSV.is_file():
    cids_df = pd.read_csv(CIDS_CSV)
    smiles_col = "SMILES_canonical" if "SMILES_canonical" in cids_df.columns else "SMILES"
    n_mida = int((cids_df["source"] == "MIDA_name_search").sum())
    print(f"CSV: {len(cids_df)} compuestos | MIDA por nombre: {n_mida}")
    if "family" in cids_df.columns:
        print(cids_df["family"].value_counts().to_string())
else:
    cids_df = None
    print("[AVISO] No se encontró pubchem_panama_cids.csv")

## 2. Cargar modelo GIN entrenado

In [ ]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"No existe {MODEL_PATH}. Ejecuta: make train-gin"
    )

with CONFIG_PATH.open(encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
model_cfg = cfg["model"]

model = GINToxicity(
    node_feat_dim=int(model_cfg["node_feat_dim"]),
    edge_feat_dim=int(model_cfg["edge_feat_dim"]),
    hidden_dim=int(model_cfg["hidden_dim"]),
    n_layers=int(model_cfg["n_layers"]),
    n_tasks=int(model_cfg["n_tasks"]),
    dropout=float(model_cfg["dropout"]),
).to(device)

try:
    state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
except TypeError:
    state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.eval()
print(f"Modelo cargado desde {MODEL_PATH}")

## 3. Predicciones multitarea

Para cada plaguicida: SMILES → grafo → 12 probabilidades de toxicidad (una por diana Tox21).

In [ ]:
def risk_level(prob: float) -> str:
    if prob > 0.7:
        return "ALTO"
    if prob > 0.4:
        return "MODERADO"
    return "BAJO"


def predict_graph(graph) -> np.ndarray:
    g = graph.to(device)
    batch = torch.zeros(g.x.size(0), dtype=torch.long, device=device)
    edge_attr = g.edge_attr if hasattr(g, "edge_attr") else None
    with torch.no_grad():
        logits = model(g.x, g.edge_index, batch, edge_attr=edge_attr)
        return torch.sigmoid(logits).squeeze().cpu().numpy()


rows = []
for name, graph in entries:
    probs = predict_graph(graph)
    max_idx = int(np.argmax(probs))
    max_task = TASK_NAMES[max_idx]
    max_prob = float(probs[max_idx])

    row = {
        "compuesto": name,
        "cid": getattr(graph, "cid", ""),
        "familia": getattr(graph, "family", ""),
        "smiles": getattr(graph, "smiles", ""),
        "tarea_critica": max_task,
        "prob_max": round(max_prob, 4),
        "alerta": risk_level(max_prob),
    }
    for task, p in zip(TASK_NAMES, probs):
        row[task] = round(float(p), 4)
    rows.append(row)

pred_df = pd.DataFrame(rows)
print(f"Predicciones generadas: {len(pred_df)} compuestos")
pred_df.head(10)

## 4. Ingredientes activos MIDA y casos prioritarios

In [ ]:
mida_lower = {n.lower() for n in MIDA_ACTIVE_INGREDIENTS}
mida_mask = pred_df["compuesto"].str.lower().isin(mida_lower)
mida_df = pred_df.loc[mida_mask].sort_values("prob_max", ascending=False)

print(f"Ingredientes MIDA con predicción: {len(mida_df)} / {len(MIDA_ACTIVE_INGREDIENTS)}")
display(mida_df[["compuesto", "familia", "tarea_critica", "prob_max", "alerta"]])

priority = [
    "Chlorpyrifos", "Atrazine", "Tebuconazole",
    "Cypermethrin", "Paraquat", "Glyphosate",
]
prio_df = pred_df[pred_df["compuesto"].isin(priority)].copy()
if prio_df.empty:
    prio_df = mida_df.head(6)

fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(prio_df))))
colors = {"ALTO": "#d62728", "MODERADO": "#ff7f0e", "BAJO": "#2ca02c"}
bar_colors = [colors.get(a, "gray") for a in prio_df["alerta"]]
ax.barh(prio_df["compuesto"], prio_df["prob_max"], color=bar_colors)
ax.axvline(0.7, color="red", ls="--", alpha=0.5, label="umbral ALTO (0.7)")
ax.axvline(0.4, color="orange", ls="--", alpha=0.5, label="umbral MODERADO (0.4)")
ax.set_xlabel("Probabilidad máxima de toxicidad")
ax.set_title("Casos prioritarios — tarea con mayor riesgo")
ax.legend(loc="lower right")
plt.tight_layout()
fig.savefig(FIG_DIR / "priority_compounds.png", bbox_inches="tight")
plt.show()

## 5. Mapa de calor del perfil de toxicidad

In [ ]:
plot_df = mida_df if len(mida_df) >= 6 else pred_df.nlargest(20, "prob_max")
heat = plot_df.set_index("compuesto")[TASK_NAMES]

plt.figure(figsize=(12, max(5, 0.35 * len(heat))))
sns.heatmap(
    heat,
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
)
plt.title("Perfil de toxicidad multitarea — plaguicidas panameños")
plt.xlabel("Diana biológica Tox21")
plt.ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "toxicity_heatmap.png", bbox_inches="tight")
plt.show()

## 6. Validación externa con etiquetas GHS

Compara predicciones del modelo con códigos H de peligro regulatorio (PubChem Hazard).
Estas etiquetas **no** se usaron para entrenar el modelo.

In [ ]:
if GHS_CSV.is_file():
    ghs_df = pd.read_csv(GHS_CSV)
    merged = pred_df.merge(ghs_df, left_on="cid", right_on="CID", how="left")

    # Umbrales orientativos: NR endocrino vs GHS reproductivo, SR genotóxico vs H340/H350
    merged["pred_endocrine"] = merged[["NR-AR", "NR-ER", "NR-ER-LBD"]].max(axis=1)
    merged["pred_genotox"] = merged[["SR-p53", "SR-AtAD5"]].max(axis=1)
    merged["pred_oxidative"] = merged["SR-ARE"]

    pairs = [
        ("pred_endocrine", "endocrine_risk", "Endocrino vs H360/H361"),
        ("pred_genotox", "genotoxic", "Genotóxico vs H340/H350"),
        ("pred_oxidative", "toxic_oral", "SR-ARE vs H300-H302"),
    ]

    print("Correlación punto-biserial (predicción continua vs etiqueta GHS binaria):\n")
    for pred_col, ghs_col, label in pairs:
        sub = merged[[pred_col, ghs_col]].dropna()
        if sub[ghs_col].nunique() < 2:
            print(f"  {label}: no hay variación en GHS")
            continue
        r = sub[pred_col].corr(sub[ghs_col], method="spearman")
        print(f"  {label}: Spearman ρ = {r:.3f} (n={len(sub)})")

    ghs_mida = merged.loc[mida_mask, ["compuesto", "tarea_critica", "prob_max", "ghs_codes", "alerta"]]
    display(ghs_mida.head(12))
else:
    print(f"No se encontró {GHS_CSV}. Ejecuta make build-panama-corpus para descargar GHS.")

## 7. Guardar resultados

In [ ]:
predictions_path = OUT_DIR / "panama_predictions.csv"
profile_path = REPORT_DIR / "panama_pesticides_profile.csv"

pred_df.to_csv(predictions_path, index=False)

profile_cols = [
    "compuesto", "cid", "familia", "smiles",
    "tarea_critica", "prob_max", "alerta",
] + TASK_NAMES
pred_df[profile_cols].to_csv(profile_path, index=False)

print(f"Predicciones: {predictions_path}")
print(f"Perfil MIDA/MINSA: {profile_path}")
print(f"Gráficos: {FIG_DIR}")

alerta_counts = pred_df["alerta"].value_counts()
print("\nDistribución de alertas (toda la corpus):")
print(alerta_counts.to_string())